# Sesión 4 · Lo que se sale de lo normal
**Análisis Estadístico de la Información — Especialización en Big Data (UCompensar)**

Este cuaderno (*notebook*) acompaña las [diapositivas de la sesión](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/diapositivas/sesion-4-lo-que-se-sale-de-lo-normal.html).
Cada celda lleva entre corchetes el mismo identificador que el código proyectado, así que puedes seguir
la clase celda a celda. La portada del curso está en https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/
y el enunciado del proyecto final, el tablero, en
[su página](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/proyecto-tablero.html).

1. **Archivo → Guardar una copia en Drive** para que tus cambios no se pierdan.
2. Ejecuta las celdas **en orden** con **Mayús + Enter**. Varias usan resultados de las anteriores.
   Si Colab avisa que el cuaderno no es de Google, elige **Ejecutar de todos modos**.
3. Si Colab se reinicia o una celda falla al repetirla, haz clic en esa celda y usa
   **Entorno de ejecución → Ejecutar anteriores**.
4. La primera celda crea los datos del caso *Tingua Domicilios S.A.S.*, una empresa **ficticia**
   (sus datos se inventan con una semilla fija y no corresponden a personas ni a empresas reales),
   y les da a las gráficas los colores del curso.

In [ ]:
#@title ▶ Crea los datos del caso (ejecútala sin modificar) { display-mode: "form" }
"""Datos del caso «Tingua Domicilios S.A.S.» (empresa FICTICIA).

Genera datos/pedidos.csv: los 12 000 pedidos que la plataforma entregó entre el
1 de enero y el 30 de junio de 2026 en cinco ciudades. Todo se inventa con una
semilla fija: los datos no corresponden a personas ni a empresas reales.

Solo usa la biblioteca estándar y, del generador aleatorio, solo random(), que
es lo único que Python garantiza igual entre versiones. Por eso la tabla sale
idéntica en Colab y en cualquier computador.

El caso sirve para todo el curso: variables de todos los tipos (sesión 1),
asimetrías y algunos atípicos (medidas y datos atípicos), una población
completa de la que se pueden sacar muestras (muestreo) y un tiempo de entrega
que depende de la distancia, la lluvia y otras variables (regresión).
"""

import csv
import math
import random
from datetime import date, timedelta
from pathlib import Path

SEMILLA = 2026
N_PEDIDOS = 12_000
INICIO = date(2026, 1, 1)
DIAS = 181  # del 1 de enero al 30 de junio de 2026

COLUMNAS = ["id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos",
            "valor_pedido", "medio_pago", "estrato", "suscriptor", "distancia_km",
            "lluvia", "temperatura_c", "tiempo_entrega_min", "calificacion"]

# ciudad: (peso, temperatura media °C, desviación, prob. de lluvia, minutos extra, factor de distancia)
CIUDADES = {
    "Bogotá":       (0.40, 14.5, 2.2, 0.30, 5.0, 1.15),
    "Medellín":     (0.22, 22.5, 2.3, 0.28, 2.0, 1.00),
    "Cali":         (0.16, 25.0, 2.3, 0.22, 0.0, 1.00),
    "Barranquilla": (0.12, 29.0, 1.8, 0.12, -1.0, 0.95),
    "Bucaramanga":  (0.10, 23.5, 2.2, 0.20, -2.0, 0.90),
}
LLUVIA_MES = {1: 0.6, 2: 0.7, 3: 1.0, 4: 1.4, 5: 1.3, 6: 0.8}

# franja: (peso, ajuste de temperatura °C, minutos extra)
FRANJAS = {
    "Mañana":   (0.15, -1.5, 0.0),
    "Almuerzo": (0.35, 1.5, 4.0),
    "Tarde":    (0.20, 1.0, 0.0),
    "Noche":    (0.30, -2.0, 3.0),
}

# categoría: (productos extra en promedio, precio medio por producto en COP, minutos de preparación)
CATEGORIAS = {
    "Restaurante": (1.3, 27_000, 3.0),
    "Mercado":     (6.0, 8_500, 4.0),
    "Farmacia":    (0.9, 21_000, 0.0),
    "Mascotas":    (1.2, 36_000, 1.0),
    "Floristería": (0.2, 88_000, 2.0),
}
# Qué se pide en cada franja (mismo orden que CATEGORIAS)
CATEGORIA_POR_FRANJA = {
    "Mañana":   (0.30, 0.35, 0.20, 0.10, 0.05),
    "Almuerzo": (0.78, 0.08, 0.07, 0.04, 0.03),
    "Tarde":    (0.40, 0.25, 0.15, 0.12, 0.08),
    "Noche":    (0.70, 0.10, 0.12, 0.05, 0.03),
}
ESTRATOS = {1: 0.06, 2: 0.22, 3: 0.33, 4: 0.20, 5: 0.11, 6: 0.08}


class _Azar:
    """Lo que hace falta del azar, construido solo sobre random()."""

    def __init__(self, semilla):
        self.r = random.Random(semilla)

    def u(self):
        return self.r.random()

    def elegir(self, opciones, pesos):
        x, acum = self.u() * sum(pesos), 0.0
        for opcion, peso in zip(opciones, pesos):
            acum += peso
            if x < acum:
                return opcion
        return opciones[-1]

    def normal(self, media=0.0, desv=1.0):
        # Box-Muller
        u1, u2 = 1.0 - self.u(), self.u()
        return media + desv * math.sqrt(-2.0 * math.log(u1)) * math.cos(2.0 * math.pi * u2)

    def exponencial(self, media):
        return -media * math.log(1.0 - self.u())

    def poisson(self, lam):
        # Knuth: suficiente para medias pequeñas
        limite, k, p = math.exp(-lam), 0, 1.0
        while True:
            p *= self.u()
            if p <= limite:
                return k
            k += 1


def _redondear(x, decimales=0):
    factor = 10 ** decimales
    return math.floor(x * factor + 0.5) / factor


def _pedido(az, dias, pesos_dias):
    fecha = az.elegir(dias, pesos_dias)
    ciudad = az.elegir(list(CIUDADES), [v[0] for v in CIUDADES.values()])
    _, t_media, t_desv, p_lluvia, min_ciudad, f_dist = CIUDADES[ciudad]
    franja = az.elegir(list(FRANJAS), [v[0] for v in FRANJAS.values()])
    _, t_franja, min_franja = FRANJAS[franja]
    categoria = az.elegir(list(CATEGORIAS), CATEGORIA_POR_FRANJA[franja])
    extra, precio, min_prep = CATEGORIAS[categoria]
    estrato = az.elegir(list(ESTRATOS), list(ESTRATOS.values()))

    suscriptor = az.u() < 0.10 + 0.06 * (estrato - 1)
    p_efectivo = 0.45 - 0.07 * (estrato - 1)
    medio_pago = az.elegir(["Efectivo", "Billetera digital", "Tarjeta"],
                           [p_efectivo, 0.25, 1 - p_efectivo - 0.25])

    n = 1 + az.poisson(extra)
    if categoria in ("Mercado", "Restaurante") and az.u() < 0.003:
        n = 20 + int(az.u() * 26)  # pedido corporativo: pocos, pero existen
    valor = n * precio * math.exp(az.normal(0, 0.22)) * (1 + 0.07 * (estrato - 3))
    valor = max(3_000, _redondear(valor / 100) * 100)

    distancia = math.exp(az.normal(math.log(3.0), 0.5)) * f_dist
    distancia = min(14.0, max(0.4, _redondear(distancia, 1)))

    lluvia = az.u() < p_lluvia * LLUVIA_MES[fecha.month]
    temperatura = _redondear(az.normal(t_media + t_franja - 1.8 * lluvia, t_desv), 1)

    tiempo = (11 + 3.1 * distancia + 0.45 * min(n, 15) + min_prep + min_franja + min_ciudad
              + 7.5 * lluvia - 2.5 * suscriptor + az.normal(0, 4) + az.exponencial(3))
    if az.u() < 0.005:
        tiempo += 55 + 95 * az.u()  # pedidos que se demoraron muchísimo
    tiempo = max(6.0, _redondear(tiempo, 1))

    latente = 6.4 - 0.065 * tiempo + 0.2 * suscriptor + az.normal(0, 0.6)
    estrellas = int(min(5, max(1, math.floor(latente + 0.5))))
    califico = az.u() < (0.80 if tiempo > 55 else 0.58)

    return {
        "fecha": fecha, "franja": franja, "ciudad": ciudad, "categoria": categoria,
        "n_productos": n, "valor_pedido": int(valor), "medio_pago": medio_pago,
        "estrato": estrato, "suscriptor": "Sí" if suscriptor else "No",
        "distancia_km": distancia, "lluvia": "Sí" if lluvia else "No",
        "temperatura_c": temperatura, "tiempo_entrega_min": tiempo,
        "calificacion": estrellas if califico else "",
    }


def crear_datos(carpeta="datos"):
    """Escribe <carpeta>/pedidos.csv y devuelve su ruta."""
    az = _Azar(SEMILLA)
    dias = [INICIO + timedelta(days=d) for d in range(DIAS)]
    pesos_dias = [1.35 if d.weekday() >= 4 else 1.0 for d in dias]  # viernes a domingo, más pedidos
    pedidos = [_pedido(az, dias, pesos_dias) for _ in range(N_PEDIDOS)]
    orden_franja = list(FRANJAS)
    pedidos.sort(key=lambda p: (p["fecha"], orden_franja.index(p["franja"])))

    ruta = Path(carpeta) / "pedidos.csv"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        escritor = csv.DictWriter(f, fieldnames=COLUMNAS)
        escritor.writeheader()
        for i, p in enumerate(pedidos, start=1):
            escritor.writerow({"id_pedido": f"P{i:05d}", **p, "fecha": p["fecha"].isoformat()})
    miles = f"{N_PEDIDOS:,}".replace(",", "\u202f")  # 12 000, con espacio fino
    print(f"Listo: {ruta} con {miles} pedidos y {len(COLUMNAS)} columnas")
    return ruta


"""Ayudas comunes a todos los notebooks del curso (van en la celda oculta, tras genera_datos.py).

Las de cada sesión están en ayudas_sesionN.py.
"""


def _aviso_version():
    """Las diapositivas se ejecutaron con pandas 2.2; con pandas 3 algunas salidas cambian."""
    import pandas as pd

    if int(pd.__version__.split(".")[0]) >= 3:
        print(f"Aviso: este Colab trae pandas {pd.__version__} y las diapositivas usan pandas 2.2. "
              "Algunas salidas se verán un poco distintas (por ejemplo, el texto sale como str "
              "y no como object).")


_aviso_version()


"""Sesión 4: estilo de las gráficas del curso y verificadores de la práctica.

Van dentro de la celda oculta del notebook, junto con genera_datos.py y ayudas.py. Cada
verificador recibe lo que el estudiante calculó, dice qué está bien y, en lo demás, da una
pista sin la respuesta. Nunca se cae: un «...» sin reemplazar o un tipo inesperado se avisa.

Tolerancias: relativa en pesos (ejercicio 1), ±0,05 en minutos (las tablas redondeadas a un
decimal valen) y ±0,005 en kilómetros en el ejercicio 3, porque ahí la media cambia apenas
centésimas. El puntaje z se acepta con las dos fórmulas de la desviación (entre n − 1 o entre n):
con 12 000 pedidos, el conteo no cambia.
"""


def _estilo_graficas():
    """Colores del curso y letras que se leen en Colab y proyectadas."""
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    plt.rcParams.update({
        "axes.prop_cycle": mpl.cycler(color=["#8A2E9E", "#F46201", "#566B7F"]),
        "figure.figsize": (6.4, 3.6),
        "figure.constrained_layout.use": True,
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#94A3B8",
        "axes.labelcolor": "#334155",
        "xtick.color": "#475569",
        "ytick.color": "#475569",
        "text.color": "#1E293B",
    })


_estilo_graficas()


def _mostrar(tabla):
    try:
        from IPython.display import display
        display(tabla)
    except ImportError:
        print(tabla.to_string())


def _original():
    """Los pedidos tal como salen del generador, o None (con aviso) si no están."""
    import pandas as pd

    try:
        return pd.read_csv("datos/pedidos.csv")
    except FileNotFoundError:
        print("✗  No encuentro datos/pedidos.csv: ejecuta primero la celda que crea los datos "
              "(Entorno de ejecución → Ejecutar anteriores)")
        return None


def _sobrantes(sobran, otros, nombres):
    """Avisa de argumentos de más o con otro nombre. True si hubo alguno."""
    if otros:
        print(f"✗  No conozco {', '.join(repr(k) for k in otros)}: usa estos nombres, en "
              f"minúsculas y sin tildes: {', '.join(n + '=' for n in nombres)}")
        return True
    if sobran:
        print(f"✗  Sobran valores: el verificador recibe solo {len(nombres)} "
              f"({', '.join(n + '=' for n in nombres)})")
        return True
    return False


def _seguro(verificador):
    """Red de seguridad: si algo imprevisto falla al revisar, se avisa sin traceback."""
    import functools

    @functools.wraps(verificador)
    def envoltura(*args, **kwargs):
        try:
            return verificador(*args, **kwargs)
        except Exception as error:   # noqa: BLE001 — el estudiante nunca ve un traceback
            print(f"✗  No pude revisar lo que entregaste ({type(error).__name__}). Revisa que cada "
                  "valor sea el resultado de un cálculo de pandas y vuelve a ejecutar la celda")
    return envoltura


_COLUMNAS = {"id_pedido", "fecha", "franja", "ciudad", "categoria", "n_productos", "valor_pedido",
             "medio_pago", "estrato", "suscriptor", "distancia_km", "lluvia", "temperatura_c",
             "tiempo_entrega_min", "calificacion"}


def _un_numero(x, n_columna=None):
    """(valor, None) si x es un número, o una serie o lista con un solo número; si no, (None, motivo)."""
    import math
    import numbers
    import numpy as np
    import pandas as pd

    if isinstance(x, pd.DataFrame):
        if x.size != 1:
            return None, f"es una tabla de {x.shape[0]} × {x.shape[1]}; hace falta un solo número"
        x = x.iloc[0, 0]
    if isinstance(x, (pd.Series, pd.Index, np.ndarray, list, tuple)):
        valores = list(np.ravel(np.asarray(x, dtype=object)))
        if n_columna is not None and len(valores) == n_columna:
            return None, "esa es la columna entera, un valor por pedido: falta resumirla"
        if len(valores) == 0:
            return None, "está vacío"
        if len(valores) > 1:
            if all(isinstance(v, float) and math.isnan(v) for v in valores):
                return None, ("salió nan: si restaste dos series, pandas las alinea por su índice "
                              "y lo que no coincide queda nan; calcula cada valor como un número")
            pista = ""
            if isinstance(x, pd.Series) and set(map(str, x.index)) & _COLUMNAS:
                pista = ": ¿elegiste la columna?"   # p. ej., pedidos.quantile(0.25, numeric_only=True)
            elif isinstance(x, pd.Series) and {"count", "mean", "std"} <= set(map(str, x.index)):
                pista = ": describe() da varias medidas; quédate solo con la que se pide"
            return None, f"tiene {len(valores)} valores; hace falta uno solo{pista}"
        x = valores[0]   # quantile([0.25]) devuelve una serie de un solo valor
    if x is None:
        return None, "está vacío (None): ¿le asignaste el resultado de un print?"
    if x is pd.NA:
        return None, "salió <NA>: revisa el cálculo (¿la columna tiene datos?)"
    if callable(x):
        return None, "es una función, no su resultado: ¿faltaron los paréntesis del final, ()?"
    if isinstance(x, (bool, np.bool_)):
        return None, "es True o False; hace falta un número"
    if isinstance(x, str):
        return None, "es texto; escríbelo como número, sin comillas y con punto decimal"
    if not isinstance(x, (numbers.Real, np.number)) or isinstance(x, np.complexfloating):
        return None, f"no es un número real (es {type(x).__name__})"
    try:
        valor = float(x)
    except (TypeError, ValueError, OverflowError):
        return None, "no es un número real"
    if math.isnan(valor):
        return None, "salió nan: revisa el cálculo (¿la columna tiene datos?)"
    if math.isinf(valor):
        return None, "salió infinito: ¿dividiste entre cero?"
    return valor, None


def _cerca(a, b, tolerancia):
    return a is not None and abs(a - b) <= tolerancia


def _relativo(a, b, tolerancia):
    """¿a está a menos de `tolerancia` (relativa) de b? Para pesos."""
    return a is not None and abs(a - b) <= tolerancia * max(abs(b), 1.0)


def _vallas(x):
    """(valla inferior, valla superior, Q1, Q3) de 1,5 RIC."""
    a, b = x.quantile(0.25), x.quantile(0.75)
    return a - 1.5 * (b - a), b + 1.5 * (b - a), a, b


@_seguro
def revisar_atipicos(*sueltos, valla_inferior=..., valla_superior=..., n_ric=..., n_z=..., **otros):
    """Ejercicio 1: vallas de 1,5 RIC de valor_pedido, pedidos fuera y pedidos con |z| > 3."""
    import pandas as pd

    nombres = ["valla_inferior", "valla_superior", "n_ric", "n_z"]
    if sueltos:
        print("✗  Pasa los valores por nombre, como en la plantilla: revisar_atipicos("
              "valla_inferior=valla_inferior, valla_superior=valla_superior, n_ric=n_ric, n_z=n_z)")
        return
    if _sobrantes((), otros, nombres):
        return
    original = _original()
    if original is None:
        return
    n = len(original)
    v = original["valor_pedido"]
    vi, vs, q1, q3 = _vallas(v)
    ric = q3 - q1
    fuera = int(((v < vi) | (v > vs)).sum())
    con_iguales = int(((v <= vi) | (v >= vs)).sum())
    solo_arriba = int((v > vs).sum())
    z1 = ((v - v.mean()) / v.std()).abs()
    z0 = ((v - v.mean()) / v.std(ddof=0)).abs()
    n_z_bien = {int((z1 > 3).sum()), int((z0 > 3).sum())}
    media, s = v.mean(), v.std()
    # z mal armado: dividido entre el RIC, o centrado con la mediana (en valor_pedido dan lo mismo)
    z_mal = {int((((v - media) / ric).abs() > 3).sum()), int((((v - v.median()) / s).abs() > 3).sum()),
             int((((v - v.median()) / ric).abs() > 3).sum())}
    # los pedidos que pueden confundirse con una valla: el mínimo, el máximo y el final de un bigote
    de_los_datos = (v.min(), v.max(), v[v >= vi].min(), v[v <= vs].max())
    # lo mismo con el tiempo de entrega: la columna de las celdas [vallas] y [puntaje_z]
    t = original["tiempo_entrega_min"]
    ti, ts, tq1, tq3 = _vallas(t)
    del_tiempo_vallas = [ti, ts, tq1, tq3, tq3 + 3 * (tq3 - tq1)]
    zt = ((t - t.mean()) / t.std()).abs()
    del_tiempo_conteos = {int(((t < ti) | (t > ts)).sum()), int((t > ts).sum()),
                          int((zt > 3).sum())}
    tol = 0.001   # relativa: en pesos

    bien = 0
    for nombre, x in [("valla_inferior", valla_inferior), ("valla_superior", valla_superior)]:
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        val, problema = _un_numero(x, n)
        if problema:
            print(f"✗  {nombre}: {problema}")
            continue
        verdad = vi if nombre == "valla_inferior" else vs
        otra = vs if nombre == "valla_inferior" else vi
        if _relativo(val, verdad, tol):
            print(f"✓  {nombre}")
            bien += 1
        elif any(_cerca(val, c, 0.051) for c in del_tiempo_vallas):
            print(f"✗  {nombre}: esa cifra es del tiempo de entrega: ¿copiaste la columna del "
                  "ejemplo? Aquí va valor_pedido")
        elif nombre == "valla_inferior" and val == 0:
            print("✗  valla_inferior: la valla puede ser negativa; el valor no. No la cambies "
                  "por cero: déjala como sale de la fórmula")
        elif nombre == "valla_inferior" and _relativo(val, -vi, tol):
            print("✗  valla_inferior: revisa el signo: la valla inferior se aleja de Q1 hacia abajo")
        elif _relativo(val, otra, tol):
            print(f"✗  {nombre}: esa es la otra valla: ¿las intercambiaste?")
        elif _relativo(val, q3 + 1.5 * q3 - q1, tol) or _relativo(val, q1 - 1.5 * q3 - q1, tol):
            print(f"✗  {nombre}: ¿faltan paréntesis? El 1,5 multiplica al RIC entero: 1.5 * (Q3 - Q1)")
        elif any(_cerca(val, c, 0.01) for c in (q1 - 1.5 * (tq3 - tq1), q3 + 1.5 * (tq3 - tq1))):
            print(f"✗  {nombre}: ¿recalculaste el RIC? Ese 1,5 RIC es el del tiempo de entrega")
        elif _relativo(val, q1, tol) or _relativo(val, q3, tol):
            print(f"✗  {nombre}: eso es un cuartil; la valla se aleja 1,5 RIC de la caja")
        elif _relativo(val, q3 + 3 * ric, tol) or _relativo(val, q1 - 3 * ric, tol):
            print(f"✗  {nombre}: esa es la valla de 3 RIC; aquí se pide la de 1,5 RIC")
        elif _relativo(val, media + 3 * s, tol) or _relativo(val, media - 3 * s, tol):
            print(f"✗  {nombre}: ese es el límite de |z| > 3; la valla usa los cuartiles y el RIC")
        elif any(_relativo(val, c, tol) for c in de_los_datos):
            print(f"✗  {nombre}: ese es un pedido de los datos (el mínimo, el máximo o el final de un "
                  "bigote); la valla se calcula con la fórmula y puede no ser ningún pedido")
        elif _relativo(val, verdad, 0.02):
            print(f"✗  {nombre}: casi: revisa los cuartiles (sin redondear) y el 1,5 RIC")
        else:
            print(f"✗  {nombre}: no cuadra: ¿usaste los cuartiles de valor_pedido y 1,5 veces su RIC?")

    for nombre, x in [("n_ric", n_ric), ("n_z", n_z)]:
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        if (isinstance(x, (pd.Series, pd.DataFrame)) and 1 < len(x) < n
                and x.index.dtype.kind in "iu"):
            # valor[condicion] o pedidos[condicion]: los pedidos mismos (no decir cuántos son)
            print(f"✗  {nombre}: son los pedidos mismos, no cuántos son: cuéntalos con .sum() "
                  "sobre la condición (cada True cuenta 1) o con len()")
            continue
        val, problema = _un_numero(x, n)
        if problema:
            print(f"✗  {nombre}: {problema}")
            continue
        if not float(val).is_integer():
            if 0 < val < 1:
                print(f"✗  {nombre}: eso parece una proporción; hace falta cuántos pedidos son: "
                      "con True y False, sum() cuenta los True")
            else:
                print(f"✗  {nombre}: un conteo es un número entero de pedidos")
            continue
        val = int(val)
        if nombre == "n_ric":
            if val == fuera:
                print("✓  n_ric")
                bien += 1
            elif val == con_iguales:
                print("✗  n_ric: ¿usaste ≥ o ≤? «Fuera» es estrictamente más allá de la valla: "
                      "< con la inferior y > con la superior")
            elif val == n - fuera or val == n - con_iguales:
                print("✗  n_ric: esos son los pedidos de dentro; se piden los de fuera")
            elif val in n_z_bien:
                print("✗  n_ric: ese conteo es el de |z| > 3; aquí van los que quedan fuera de las vallas")
            elif val in del_tiempo_conteos:
                print("✗  n_ric: ese conteo es del tiempo de entrega: ¿copiaste la columna del ejemplo?")
            elif val == 0:
                print("✗  n_ric: salió cero. Un pedido queda fuera si está por debajo de la inferior o "
                      "por encima de la superior: cuenta los dos lados y únelos con | («o»), no con & («y»)")
            elif val == n:
                print("✗  n_ric: son todos los pedidos: ¿comparaste valor_pedido con sus propias vallas?")
            else:
                print("✗  n_ric: no cuadra: cuenta los pedidos por debajo de la valla inferior o por "
                      "encima de la superior")
        else:
            if val in n_z_bien:
                print("✓  n_z")
                bien += 1
            elif val in (fuera, con_iguales):
                print("✗  n_z: ese conteo es el de las vallas; aquí van los de |z| > 3")
            elif val in del_tiempo_conteos:
                print("✗  n_z: ese conteo es del tiempo de entrega: ¿copiaste la columna del ejemplo?")
            elif val == 0:
                print("✗  n_z: salió cero: z se calcula con la media y la desviación de valor_pedido")
            elif val == n:
                print("✗  n_z: marca todos los pedidos: ¿faltan los paréntesis? z = (valor − media) / desviación")
            elif val in {n - k for k in n_z_bien}:
                print("✗  n_z: esos son los de dentro (|z| ≤ 3); se piden los de |z| > 3")
            elif val in z_mal:
                print("✗  n_z: z se centra con la media y se divide entre la desviación estándar, "
                      "std(): ¿usaste el RIC o la mediana?")
            elif val > max(n_z_bien):
                print("✗  n_z: marca de más: revisa el límite (3), el centro (la media) y el divisor "
                      "(la desviación, std())")
            else:
                print("✗  n_z: no cuadra: z = (valor − media) / desviación, y se cuentan los de |z| > 3")

    if bien == 4:
        tabla = pd.DataFrame({"valor_pedido": [vi, vs, fuera, (v < vi).sum(), (v > vs).sum(),
                                               max(n_z_bien)]},
                             index=["valla inferior (pesos)", "valla superior (pesos)",
                                    "fuera de las vallas", "· por debajo de la inferior",
                                    "· por encima de la superior", "con |z| > 3"])
        _mostrar(tabla.round(0).astype(int))
        print("✓  Los cuatro cuadran. Responde abajo: ¿por qué z marca menos pedidos que las vallas?")


def _texto(etiquetas):
    """Etiquetas comparadas como texto (las categóricas y los números también)."""
    return [str(e) for e in etiquetas]


@_seguro
def revisar_dinamica(tabla=..., *sobran, **otros):
    """Ejercicio 2: tabla dinámica de la mediana del tiempo por ciudad (filas) y franja (columnas).

    Acepta la tabla transpuesta, márgenes «All» o «Total», etiquetas categóricas (se comparan como
    texto), una serie con dos niveles (con la pista de unstack) y valores redondeados a un decimal.
    """
    import numbers
    import numpy as np
    import pandas as pd

    if _sobrantes(sobran, otros, ["tabla"]):
        return
    if tabla is Ellipsis:
        print("·  tabla: todavía tiene ...")
        return
    original = _original()
    if original is None:
        return
    n = len(original)
    franjas = ["Mañana", "Almuerzo", "Tarde", "Noche"]
    original["franja"] = pd.Categorical(original["franja"], categories=franjas, ordered=True)
    ciudades = sorted(original["ciudad"].unique())
    grupos = original.groupby(["ciudad", "franja"], observed=True)["tiempo_entrega_min"]

    def como_tabla(serie):
        return serie.unstack().reindex(index=ciudades, columns=franjas)

    verdad = como_tabla(grupos.median())
    otras = {"tiene la media del tiempo": como_tabla(grupos.mean()),
             "cuenta pedidos": como_tabla(grupos.size().astype(float)),
             "tiene la suma del tiempo": como_tabla(grupos.sum()),
             "tiene el máximo del tiempo": como_tabla(grupos.max()),
             "tiene el mínimo del tiempo": como_tabla(grupos.min())}
    margenes = {"All", "Total"}
    pista_serie = False

    x = tabla
    # --- sin resumir: una fila por pedido, o varias por grupo (unstack fallaría) ---
    if isinstance(x, (pd.Series, pd.DataFrame)):
        if len(x) == n:
            print("✗  tabla: tiene una fila por pedido: falta resumir cada grupo con la mediana "
                  "(pd.pivot_table con values=, index=, columns= y aggfunc=)")
            return
        if isinstance(x.index, pd.MultiIndex) and x.index.has_duplicates:
            print("✗  tabla: tiene varias filas por cada ciudad y franja: falta resumir cada grupo "
                  "con la mediana")
            return
    # --- lo que no es una tabla ---
    if isinstance(x, pd.Series):
        if isinstance(x.index, pd.MultiIndex) and x.index.nlevels == 2:
            x = x.unstack()
            pista_serie = True
        elif isinstance(x.index, pd.MultiIndex):
            print(f"✗  tabla: cruza {x.index.nlevels} niveles; la tabla dinámica cruza solo dos: "
                  "las ciudades y las franjas")
            return
        else:
            print("✗  tabla: tiene una sola dimensión; la tabla dinámica cruza dos: las ciudades "
                  "y las franjas")
            return
    if not isinstance(x, pd.DataFrame):
        if x is None:
            print("✗  tabla: está vacía (None): ¿le asignaste el resultado de un print?")
        elif isinstance(x, str):
            print("✗  tabla: es texto; hace falta la tabla que devuelve pd.pivot_table")
        elif callable(x):
            print("✗  tabla: es una función, no su resultado: ¿faltaron los paréntesis o los "
                  "argumentos de pd.pivot_table?")
        elif type(x).__name__.endswith("GroupBy"):
            print("✗  tabla: es el agrupamiento, todavía sin la medida: falta calcularla en cada grupo")
        elif isinstance(x, (numbers.Number, np.number)):
            print("✗  tabla: es un solo número; hace falta una tabla de ciudades por franjas")
        elif isinstance(x, (list, tuple, dict, np.ndarray)):
            print("✗  tabla: no es una tabla de pandas: usa pd.pivot_table, que trae las "
                  "ciudades y las franjas como etiquetas")
        else:
            print(f"✗  tabla: tiene que ser una tabla de pandas (es {type(x).__name__})")
        return
    # --- una tabla: la forma ---
    if isinstance(x.index, pd.MultiIndex) and x.index.nlevels == 2 and x.shape[1] == 1:
        x = x.iloc[:, 0].unstack()   # groupby([...])[["col"]].median(): una columna, dos niveles
        pista_serie = True
    funciones = {"median", "mean", "count", "size", "sum", "min", "max", "std", "var", "nunique"}
    while isinstance(x.columns, pd.MultiIndex):
        nivel0 = set(_texto(x.columns.get_level_values(0)))
        if len(nivel0) == 1 and nivel0 <= _COLUMNAS | funciones:
            x = x.droplevel(0, axis=1)   # values=[...] o aggfunc=[...], entre corchetes
        elif nivel0 & _COLUMNAS:
            print("✗  tabla: resume varias columnas a la vez: ¿falta values=? Indica la medida que "
                  "se resume")
            return
        elif nivel0 <= funciones:
            print("✗  tabla: resume con varias funciones a la vez; en aggfunc= deja solo la que se pide")
            return
        else:
            print("✗  tabla: sus columnas tienen varios niveles: ¿pusiste dos dimensiones en "
                  "columns=? Van una en index= y otra en columns=")
            return
    if isinstance(x.index, pd.MultiIndex):
        cols = set(_texto(x.columns))
        if {"count", "mean", "50%"} <= cols:
            print("✗  tabla: describe() da varias medidas; aquí va solo la mediana, con la franja "
                  "en las columnas (pd.pivot_table lo hace de una vez)")
        elif x.index.nlevels == 2 and cols <= _COLUMNAS | funciones:
            print("✗  tabla: tiene varias medidas por grupo; aquí va solo la mediana del tiempo, con "
                  "la franja en las columnas (pd.pivot_table lo hace de una vez)")
        else:
            print("✗  tabla: sus filas tienen varios niveles: ¿pusiste dos dimensiones en index=? Van "
                  "una en index= y otra en columns=")
        return
    if x.size == 0:
        print("✗  tabla: está vacía")
        return
    columnas_txt = set(_texto(x.columns))
    if "ciudad" in columnas_txt or "franja" in columnas_txt:
        print("✗  tabla: una dimensión quedó como columna (¿usaste reset_index?); déjala en el "
              "índice, como sale de pd.pivot_table")
        return
    filas = [e for e in _texto(x.index) if e not in margenes]
    cols = [e for e in _texto(x.columns) if e not in margenes]
    hay_margenes = len(filas) < len(x.index) or len(cols) < len(x.columns)
    if len(set(filas)) != len(filas) or len(set(cols)) != len(cols):
        print("✗  tabla: tiene etiquetas repetidas; cada ciudad y cada franja deben aparecer una vez")
        return
    if sorted(filas) == sorted(franjas) and sorted(cols) == sorted(ciudades):
        x = x.T   # transpuesta: también vale
        filas, cols = cols, filas
    if not (sorted(filas) == sorted(ciudades) and sorted(cols) == sorted(franjas)):
        dims = set(filas) | set(cols)
        faltan = ([c for c in ciudades if c not in dims] if set(ciudades) & dims else []) + \
                 ([f for f in franjas if f not in dims] if set(franjas) & dims else [])
        if dims & {"Sí", "No"}:
            print("✗  tabla: una de sus dimensiones es la lluvia, como en la celda [dinamica]; aquí "
                  "se cruzan la ciudad y la franja")
        elif faltan and (dims - set(ciudades) - set(franjas)) == set():
            print(f"✗  tabla: le falta{'n' if len(faltan) > 1 else ''} {', '.join(faltan)}: ¿filtraste "
                  "los pedidos o quitaste columnas? La tabla va con todos")
        elif set(filas) == set(ciudades) or set(cols) == set(ciudades):
            print("✗  tabla: las ciudades están; la otra dimensión tiene que ser la franja")
        elif set(filas) == set(franjas) or set(cols) == set(franjas):
            print("✗  tabla: las franjas están; la otra dimensión tiene que ser la ciudad")
        elif dims & _COLUMNAS:
            print("✗  tabla: sus etiquetas son nombres de columnas: ¿pasaste la medida en index= o "
                  "en columns=? La medida va en values=")
        else:
            print("✗  tabla: sus filas y columnas tienen que ser las cinco ciudades y las cuatro "
                  "franjas: revisa index= y columns=")
        return
    # --- los valores ---
    sin_margenes = x.loc[[e not in margenes for e in _texto(x.index)],
                         [e not in margenes for e in _texto(x.columns)]]
    if pd.api.types.is_bool_dtype(sin_margenes.stack()):
        print("✗  tabla: sus valores son True o False; hace falta la mediana del tiempo")
        return
    if any(isinstance(e, str) for e in sin_margenes.to_numpy().ravel()):
        print("✗  tabla: sus valores son texto (¿astype(str)?); déjalos como números")
        return
    try:
        valores = sin_margenes.apply(pd.to_numeric, errors="raise").astype(float)
    except (TypeError, ValueError):
        print("✗  tabla: sus valores no son números (¿astype(str)?); déjalos como números")
        return
    valores.index = _texto(valores.index)
    valores.columns = _texto(valores.columns)
    valores = valores.reindex(index=ciudades, columns=franjas)
    if valores.isna().any().any():
        print("✗  tabla: tiene celdas vacías (nan): ¿filtraste los pedidos antes de construirla?")
        return

    def igual(a, b, tolerancia=0.051):
        return bool((a - b).abs().max().max() <= tolerancia)

    if igual(valores, verdad):
        print("✓  tabla" + (" (con márgenes, que no se revisan)" if hay_margenes else ""))
        if pista_serie:
            print("·  Las dos dimensiones quedaron en las filas: con .unstack() las franjas pasan "
                  "a las columnas. pd.pivot_table la construye así de una vez")
        _mostrar(verdad.round(1))
        print("✓  Cuadra. Responde abajo: ¿la franja más lenta es la misma en todas las ciudades?")
        return
    for nombre, otra in otras.items():
        if igual(valores, otra):
            print(f"✗  tabla: esa tabla {nombre}; se pide la mediana del tiempo (aggfunc=)")
            return
    if igual(valores, verdad, 0.51) and bool((valores == valores.round()).all().all()):
        print("✗  tabla: casi: le quitaste los decimales; deja al menos uno")
        return
    numericas = original.select_dtypes("number").drop(columns="tiempo_entrega_min")
    for col in numericas.columns:
        mediana_otra = como_tabla(original.groupby(["ciudad", "franja"], observed=True)[col].median())
        if igual(valores, mediana_otra.astype(float), 1e-6):
            print("✗  tabla: esas medianas son de otra columna: ¿values= es tiempo_entrega_min?")
            return
    print("✗  tabla: no cuadra: revisa la medida (values=), la función (aggfunc=) y que estén "
          "todos los pedidos")


@_seguro
def revisar_tope(*sueltos, mediana_30=..., media_30=..., **otros):
    """Ejercicio 3 (opcional): mediana y media de la distancia si los 20 pedidos de 14 km fueran de 30."""
    import pandas as pd

    if sueltos:
        print("✗  Pasa los valores por nombre, como en la plantilla: "
              "revisar_tope(mediana_30=mediana_30, media_30=media_30)")
        return
    if _sobrantes((), otros, ["mediana_30", "media_30"]):
        return
    original = _original()
    if original is None:
        return
    d = original["distancia_km"]
    n = len(d)
    d30 = d.where(d != 14, 30)
    verdad = {"mediana_30": d30.median(), "media_30": d30.mean()}
    con_14 = {"mediana": d.median(), "media": d.mean()}
    sin_tope = d[d != 14]
    tol = 0.005
    bien = 0

    for nombre, x in [("mediana_30", mediana_30), ("media_30", media_30)]:
        if x is Ellipsis:
            print(f"·  {nombre}: todavía tiene ...")
            continue
        val, problema = _un_numero(x, n)
        if problema:
            print(f"✗  {nombre}: {problema}")
            continue
        if nombre == "mediana_30":
            if _cerca(val, verdad["mediana_30"], tol):
                print("✓  mediana_30")
                bien += 1
            elif _cerca(val, verdad["media_30"], tol) or _cerca(val, con_14["media"], tol):
                print("✗  mediana_30: eso parece una media; la mediana es el valor del medio")
            elif _cerca(val, 30, tol) or _cerca(val, 14, tol):
                print("✗  mediana_30: esa es la distancia de los 20 pedidos; la mediana es la de "
                      "los 12 000")
            elif _cerca(val, sin_tope.median(), tol):
                print("✗  mediana_30: ¿quitaste los 20 pedidos? La idea es cambiarles la distancia, "
                      "no borrarlos")
            else:
                print("✗  mediana_30: no cuadra: es la mediana de la copia, con los 12 000 pedidos")
        else:
            if _cerca(val, verdad["media_30"], tol):
                print("✓  media_30")
                bien += 1
            elif _cerca(val, con_14["media"], tol):
                print("✗  media_30: esa es la media con los 14 km: ¿la copia tiene 30 donde había "
                      "14? replace devuelve una copia: guárdala en distancia_30")
            elif _cerca(val, sin_tope.mean(), tol):
                print("✗  media_30: ¿quitaste los 20 pedidos? La idea es cambiarles la distancia, "
                      "no borrarlos")
            elif _cerca(val, verdad["mediana_30"], tol):
                print("✗  media_30: eso parece la mediana; la media suma y divide")
            elif any(_cerca(val, round(verdad["media_30"], k), 1e-9) for k in (0, 1)):
                print("✗  media_30: casi: aquí la diferencia está en las centésimas; deja al "
                      "menos dos decimales")
            elif _cerca(val, 30, tol) or _cerca(val, 14, tol):
                print("✗  media_30: esa es la distancia de los 20 pedidos; la media es la de los "
                      "12 000, con esos 20 cambiados")
            elif val > verdad["media_30"]:
                print("✗  media_30: salió de más: ¿cambiaste solo los pedidos de exactamente 14 km, "
                      "y por 30?")
            else:
                print("✗  media_30: no cuadra: es la media de la copia, con los 12 000 pedidos")

    # el enunciado pide la copia «sin cambiar pedidos»: se mira la tabla del cuaderno
    try:
        from IPython import get_ipython
        cuaderno = get_ipython().user_ns.get("pedidos") if get_ipython() is not None else None
    except ImportError:
        cuaderno = None
    if (isinstance(cuaderno, pd.DataFrame) and "distancia_km" in cuaderno.columns
            and len(cuaderno) == n
            and not cuaderno["distancia_km"].reset_index(drop=True).equals(d)):
        print("✗  pedidos: su distancia_km ya no es la original: la copia se hace sin tocar pedidos. "
              "Vuelve a ejecutar la celda [cargar] y guarda el cambio solo en distancia_30")
        return
    if bien == 2:
        tabla = pd.DataFrame({"con 14 km": [con_14["mediana"], con_14["media"]],
                              "con 30 km": [verdad["mediana_30"], verdad["media_30"]]},
                             index=["mediana", "media"])
        _mostrar(tabla.round(3))
        print("✓  Las dos cuadran. Responde abajo: ¿qué cambia y qué no? ¿Qué te dice eso de un "
              "dato censurado?")

# Crea datos/pedidos.csv: los mismos 12 000 pedidos de las sesiones 1 a 3
ruta = crear_datos("datos")

## 1 · Los datos, con sus escalas

Cargamos los pedidos y declaramos las escalas, como en las sesiones anteriores.

In [ ]:
# [cargar]
import pandas as pd
import matplotlib.pyplot as plt

pedidos = pd.read_csv("datos/pedidos.csv")
# Las escalas que declaramos en la sesión 1
franjas = ["Mañana", "Almuerzo", "Tarde", "Noche"]
pedidos["franja"] = pd.Categorical(
    pedidos["franja"], categories=franjas, ordered=True)
pedidos["estrato"] = pd.Categorical(
    pedidos["estrato"], categories=[1, 2, 3, 4, 5, 6], ordered=True)
pedidos["calificacion"] = pd.Categorical(
    pedidos["calificacion"], categories=[1, 2, 3, 4, 5], ordered=True)
pedidos.shape

### Diccionario de datos (el mismo de la sesión 1)

| Columna | Qué registra | Valores o unidad |
|---|---|---|
| `id_pedido` | Código del pedido | P00001, P00002… |
| `fecha` | Día del pedido | del 1 de enero al 30 de junio de 2026 |
| `franja` | Momento del día en que se pidió | Mañana, Almuerzo, Tarde, Noche |
| `ciudad` | Ciudad de entrega | Bogotá, Medellín, Cali, Barranquilla, Bucaramanga |
| `categoria` | Tipo de comercio que despacha | Restaurante, Mercado, Farmacia, Mascotas, Floristería |
| `n_productos` | Unidades pedidas | conteo |
| `valor_pedido` | Valor de los productos, sin domicilio ni propina | pesos (COP) |
| `medio_pago` | Cómo se pagó | Efectivo, Tarjeta, Billetera digital |
| `estrato` | Estrato de la **dirección de entrega** (lo asigna el municipio a la vivienda) | 1 a 6 |
| `suscriptor` | Si quien pide paga la suscripción de Tingua | Sí / No |
| `distancia_km` | Distancia por vía del comercio a la dirección | kilómetros; la app registra hasta 14: una ruta más larga queda como 14 |
| `lluvia` | Si llovía en la zona durante la entrega | Sí / No |
| `temperatura_c` | Temperatura en la zona al momento de la entrega | grados Celsius |
| `tiempo_entrega_min` | Minutos desde que se confirma el pedido hasta que llega | minutos |
| `calificacion` | Estrellas que dio quien pidió; vacía si no calificó | 1 a 5 |

## 2 · Los atípicos: ¿qué pedidos se salen de lo normal?

Un **atípico** es un dato que queda lejos de los demás. Ante cada candidato hacemos dos preguntas:

- **¿Por qué está lejos?** Puede ser un error, un valor real y raro, o un dato censurado.
- **¿Lejos respecto a qué?** De todos los pedidos o de los de su grupo.

### La regla de 1,5 RIC (las vallas de Tukey)

Las **vallas** se ponen a 1,5 RIC de la caja: $Q_1 - 1{,}5\,\text{RIC}$ por debajo y
$Q_3 + 1{,}5\,\text{RIC}$ por encima. Un pedido queda **fuera** si está **estrictamente** más allá
de una valla: menor que la inferior o mayor que la superior.

In [ ]:
# [vallas]
tiempo = pedidos["tiempo_entrega_min"]
q1, q3 = tiempo.quantile(0.25), tiempo.quantile(0.75)
ric = q3 - q1
valla_inf = q1 - 1.5 * ric
valla_sup = q3 + 1.5 * ric
print(f"Vallas: {valla_inf:.2f} y {valla_sup:.2f} min")
# «Fuera» es estrictamente más allá: < y >, no <= ni >=
print("Por debajo:", (tiempo < valla_inf).sum(), "pedidos")
print("Por encima:", (tiempo > valla_sup).sum(), "pedidos")

Con 3 RIC en lugar de 1,5, la valla marca solo los **extremos**.

In [ ]:
# [extremos]
valla_ext = q3 + 3 * ric
print(f"Valla de 3 RIC: {valla_ext:.1f} min ·",
      (tiempo > valla_ext).sum(), "pedidos por encima")

### Las vallas en el diagrama de caja

El diagrama de caja de la sesión 3 usa esta misma regla. La **caja** va de $Q_1$ a $Q_3$ y la línea
es la mediana. Los **bigotes** no llegan hasta la valla, sino hasta el último pedido que **no** la
pasa: solo coinciden con ella si un pedido vale justo la valla. Un bigote llega al mínimo o al
máximo solo del lado en que no hay puntos sueltos. Los **puntos sueltos** son los que pasan la
valla: los candidatos. Varios pedidos con el mismo valor se dibujan como un solo punto.

In [ ]:
# [caja_tiempo]
estilo_caja = dict(          # los colores de la caja, para todas las cajas
    widths=0.5, patch_artist=True,
    boxprops={"facecolor": "#EFE3F2", "edgecolor": "#8A2E9E"},
    medianprops={"color": "#3A0F47", "linewidth": 2.5},
    whiskerprops={"color": "#8A2E9E"},
    capprops={"color": "#8A2E9E"},
    flierprops={"marker": ".", "markersize": 4,
                "markerfacecolor": "#8A2E9E", "markeredgecolor": "none"})
fig, ax = plt.subplots(figsize=(5.2, 4.2))
caja = ax.boxplot(tiempo, **estilo_caja)
for valla in [valla_inf, valla_sup]:       # las dos vallas, con su valor
    ax.axhline(valla, color="#F46201", linewidth=2, linestyle="--")
    ax.text(1.48, valla + 2, f"valla: {valla:.2f}", ha="right")
ax.set_xticks([1], ["Todos los pedidos"])
ax.set_ylabel("Minutos")
# Los bigotes llegan al último pedido que no pasa cada valla
bigotes = [linea.get_ydata()[1] for linea in caja["whiskers"]]
print(f"Bigotes: de {bigotes[0]:.1f} a {bigotes[1]:.1f} min")
sueltos = caja["fliers"][0].get_ydata()
print("Fuera de los bigotes:", len(sueltos), "pedidos en",
      pd.Series(sueltos).nunique(), "puntos distintos")
ax.set_title(f"{len(sueltos)} pedidos pasan las vallas")   # un título que afirma
plt.show()

### El puntaje z

El **puntaje z** dice a cuántas desviaciones estándar de la media está un pedido:

$$z = \frac{x - \bar{x}}{s}$$

Por convención, se marca un pedido si $|z| > 3$: si está a más de tres desviaciones de la media,
por encima o por debajo. Aquí z solo mide distancias; su relación con la distribución normal llega
en la sesión 6.

In [ ]:
# [puntaje_z]
media, s = tiempo.mean(), tiempo.std()
z = (tiempo - media) / s           # a cuántas desviaciones de la media
print("|z| > 3:", (z.abs() > 3).sum(), "pedidos")
marcados = tiempo[z.abs() > 3]
print("¿Todos por encima de la valla?", bool((marcados > valla_sup).all()))
print(f"Media - 3s = {media - 3 * s:.1f} min")   # ningún tiempo es negativo

### ¿Por qué z marca menos? El enmascaramiento

z usa la media y la desviación, y los propios candidatos las inflan: así se esconden.

In [ ]:
# [enmascaramiento]
fuera = (tiempo < valla_inf) | (tiempo > valla_sup)
print(f"s con todos los pedidos: {s:.2f} min")
print(f"s sin los {fuera.sum()} candidatos: {tiempo[~fuera].std():.2f} min")

### Las dos reglas en el histograma

La franja sombreada va de la valla superior a $z = 3$: son los pedidos que la valla marca y z no.

In [ ]:
# [histograma_vallas]
z3 = media + 3 * s                 # el tiempo donde z vale 3
entre = ((tiempo > valla_sup) & (tiempo <= z3)).sum()   # la valla sí, z no
mas_alla = (tiempo > z3).sum()                          # las dos
ax = tiempo.plot.hist(bins=range(0, 210, 5), color="#8A2E9E", alpha=0.45,
                      edgecolor="white")
ax.axvspan(valla_sup, z3, color="#F46201", alpha=0.12)
ax.axvline(valla_sup, color="#F46201", linewidth=2.5, linestyle="--")
ax.axvline(z3, color="#3A0F47", linewidth=2.5, linestyle=":")
alto = ax.get_ylim()[1]
ax.text(valla_sup - 1.5, alto * 0.95, f"valla\n{valla_sup:.2f}",
        ha="right", va="top")
ax.text(z3 + 1.5, alto * 0.95, f"z = 3\n{z3:.1f}", ha="left", va="top")
ax.text((valla_sup + z3) / 2, alto * 0.5, str(entre), ha="center",
        fontweight="bold", color="#B84A00")
ax.text(z3 + 2, alto * 0.35,
        f"{mas_alla} más allá,\nhasta {tiempo.max():.1f} min", ha="left")
ax.set_xlim(0, 140)   # la cola sigue más allá de 200 min
ax.set_title(f"La valla superior marca {entre + mas_alla}; z = 3, solo {mas_alla}")
ax.set_xlabel("Minutos")
ax.set_ylabel("Pedidos")
plt.show()

### ¿Son raros todos los que marca la regla?

En `n_productos` la regla marca muchos pedidos. Antes de creerle, miremos la gráfica: a la
izquierda, la caja; en el centro, los pedidos de 1 a 17 productos, con los de fuera en un color
más claro; a la derecha, ampliados los de más de 14 productos.

In [ ]:
# [productos_barras]
productos = pedidos["n_productos"]
p1, p3 = productos.quantile(0.25), productos.quantile(0.75)
valla_p = p3 + 1.5 * (p3 - p1)
print("Valla superior:", valla_p, "· fuera:", (productos > valla_p).sum(),
      "· justo en la valla:", (productos == valla_p).sum())
conteo = productos.value_counts().sort_index()
fig, (caja_p, izq, der) = plt.subplots(1, 3, width_ratios=[1, 3, 2],
                                       figsize=(8.4, 3.6))
sueltos = caja_p.boxplot(productos, **estilo_caja)["fliers"][0].get_ydata()
print("En la caja:", len(sueltos), "pedidos en",
      pd.Series(sueltos).nunique(), "puntos distintos")
caja_p.set_xticks([1], ["Caja"])
caja_p.set_title("La caja", fontsize=12)
caja_p.set_ylabel("Productos")
fuera_p = conteo.index > valla_p       # «fuera» es estrictamente más de 6
izq.axvline(valla_p, color="#F46201", linewidth=2.5, linestyle="--",
            zorder=0, label=f"Valla superior: {valla_p:.0f}")   # detrás
izq.bar(conteo.index[~fuera_p], conteo.values[~fuera_p], color="#8A2E9E")
izq.bar(conteo.index[fuera_p], conteo.values[fuera_p], color="#8A2E9E",
        alpha=0.45, label=f"Fuera: más de {valla_p:.0f}")
izq.set_xlim(0, 18)                    # lo de más de 14, en la ampliación
izq.legend()
izq.set_title(f"La regla marca {(productos > valla_p).sum()} pedidos")
izq.set_xlabel("Productos")
izq.set_ylabel("Pedidos")
grandes = conteo[conteo.index > 14]
corp = grandes.index >= 20             # el grupo aparte
der.bar(grandes.index[~corp], grandes.values[~corp], color="#8A2E9E",
        alpha=0.45)                    # fuera, como en el centro
der.bar(grandes.index[corp], grandes.values[corp], color="#3A0F47")
der.text(32, 4.6, f"{(productos >= 20).sum()} pedidos\nde 20 a {productos.max()}",
         ha="center")
der.set_title("Ampliado: más de 14", fontsize=12)
der.set_xlabel("Productos")
plt.show()

### Los pedidos lentos son dos grupos

Los 240 pedidos por encima de la valla superior no son todos iguales. Partámoslos en
**moderados** (hasta la valla de 3 RIC) y **extremos** (más allá).

In [ ]:
# [lentos]
lentos = pedidos[tiempo > valla_sup].copy()
lentos["grupo"] = pd.cut(lentos["tiempo_entrega_min"],
                         bins=[valla_sup, valla_ext, 250],
                         labels=["moderado", "extremo"])
lentos["llueve"] = (lentos["lluvia"] == "Sí") * 100      # 100 o 0
lentos["bogota"] = (lentos["ciudad"] == "Bogotá") * 100
resumen = lentos.groupby("grupo", observed=True).agg(
    pedidos=("id_pedido", "count"),
    km_mediana=("distancia_km", "median"),
    pct_lluvia=("llueve", "mean"),
    pct_bogota=("bogota", "mean"))
resumen.round(2)

¿Y la calificación de las demoras extremas? Contamos estrellas, sin promediarlas.

In [ ]:
# [extremos_calificacion]
extremos = pedidos[tiempo > valla_ext]
calificados = extremos["calificacion"].dropna()
print("Extremos calificados:", len(calificados),
      "· con 1 estrella:", (calificados == 1).sum())
# El pedido más lento de todos (la pregunta de la sesión 1)
columnas = ["id_pedido", "ciudad", "distancia_km", "lluvia",
            "tiempo_entrega_min", "calificacion"]
pedidos.loc[tiempo.idxmax(), columnas]

### ¿Atípico respecto a qué?

La valla se puede calcular con todos los pedidos (la valla **global**) o con los de cada ciudad
(la valla **propia**). Un mismo pedido puede ser atípico en su ciudad y normal en el total.

In [ ]:
# [valla_ciudad]
ciudad = pedidos["ciudad"]
grupos = tiempo.groupby(ciudad)
q1c, q3c = grupos.quantile(0.25), grupos.quantile(0.75)
valla_c = q3c + 1.5 * (q3c - q1c)            # una valla por ciudad
lento_global = tiempo > valla_sup
lento_propia = tiempo > ciudad.map(valla_c)  # la valla de su ciudad
pd.DataFrame({
    "valla propia": valla_c,
    "lentos (global)": lento_global.groupby(ciudad).sum(),
    "lentos (propia)": lento_propia.groupby(ciudad).sum(),
}).round(2)

Un pedido de 55 min en Barranquilla pasa la valla de su ciudad, pero no la global. ¿Cuántos hay así?

In [ ]:
# [solo_ciudad]
# Atípicos en Barranquilla con la valla propia, pero no con la global
solo_propia = (ciudad == "Barranquilla") & lento_propia & ~lento_global
print("Barranquilla, atípicos solo en su ciudad:", solo_propia.sum())

Barranquilla y Bucaramanga tenían casi el mismo cuerpo (sesión 3). ¿Cuántos pedidos pasan la valla
de su ciudad, y cuánto se alejan de ella?

In [ ]:
# [alejan]
for nombre in ["Barranquilla", "Bucaramanga"]:
    x = tiempo[ciudad == nombre]
    exceso = x[x > valla_c[nombre]] - valla_c[nombre]   # cuánto se pasa
    print(f"{nombre}: {len(exceso)} pasan su valla;",
          f"se alejan {exceso.median():.1f} min en mediana",
          f"y {exceso.max():.1f} el más lejano")

Las cajas por ciudad de la sesión 3 usan la valla **propia** de cada ciudad. Encima va la valla
**global**: la misma línea para todas.

In [ ]:
# [cajas_ciudad]
orden = valla_c.sort_values().index     # de menor a mayor valla
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.boxplot([tiempo[ciudad == c] for c in orden], **estilo_caja)
ax.axhline(valla_sup, color="#F46201", linewidth=2, linestyle="--")
ax.set_xticks(range(1, len(orden) + 1), orden)
# Sin leyenda: taparía los puntos más altos (el de 200.9 min)
ax.set_title(f"Por ciudad; en naranja, la valla global ({valla_sup:.2f} min)")
ax.set_ylabel("Minutos")
plt.show()

### El tope de 14 km: un dato censurado

El diccionario de datos lo dice: la app registra hasta 14 km, y una ruta más larga queda como 14.
Un «14» significa «14 o más»: es un **dato censurado**.

In [ ]:
# [tope]
distancia = pedidos["distancia_km"]
distancia.value_counts().sort_index().tail()

In [ ]:
# [distancia_tope]
lejos = distancia[distancia > 8]
conteo = lejos.value_counts().sort_index()
fig, (caja_d, ax) = plt.subplots(1, 2, width_ratios=[1, 4])
sueltos = caja_d.boxplot(distancia, **estilo_caja)["fliers"][0].get_ydata()
print("En la caja:", len(sueltos), "pedidos en",
      pd.Series(sueltos).nunique(), "puntos distintos")
caja_d.set_xticks([1], ["Caja"])
caja_d.set_title("La caja", fontsize=12)
caja_d.set_ylabel("Kilómetros")
ax.bar(conteo.index, conteo.values, width=0.08, color="#8A2E9E",
       label="Pedidos a cada distancia")
ax.bar([14], [conteo[14.0]], width=0.08, color="#F46201",
       label="Exactamente 14.0 km")
ax.legend()
ax.set_title(f"{conteo[14.0]} pedidos en el tope de 14 km")
ax.set_xlabel("Kilómetros")
ax.set_ylabel("Pedidos")
plt.show()

### Qué hacer con cada candidato

| Pregunta | Si la respuesta es sí | Qué se hace |
|---|---|---|
| ¿Es imposible o choca con otras columnas? | Es un **error** (seguro o probable) | Se corrige en la fuente o se marca como faltante |
| ¿Está en el tope del registro? | Es un dato **censurado** | Se marca; no se borra ni se trata como exacto |
| ¿Es posible, coherente y no está en un tope? | Es un atípico **real** | Se conserva, se investiga y se informa con medidas robustas |

Un valor posible también puede ser un error (un 20,9 escrito 200,9): por eso se mira si es
coherente con las demás columnas. Primero, ¿hay errores evidentes en Tingua?

In [ ]:
# [errores]
print("Filas repetidas:", pedidos.duplicated().sum())
print("Códigos repetidos:", pedidos["id_pedido"].duplicated().sum())
cantidades = ["n_productos", "valor_pedido", "distancia_km",
              "tiempo_entrega_min"]
print("Valores ≤ 0:", (pedidos[cantidades] <= 0).sum().sum())

¿Y si borramos los candidatos y recalculamos las vallas?

In [ ]:
# [cadena]
resto = tiempo[~fuera]                 # sin los candidatos de la valla
a, b = resto.quantile(0.25), resto.quantile(0.75)
nuevos = (resto < a - 1.5 * (b - a)) | (resto > b + 1.5 * (b - a))
print("Borrados:", fuera.sum(), "· atípicos nuevos:", nuevos.sum())

## 3 · Exploración multidimensional

Un **cubo** de datos cruza **dimensiones** (por dónde se corta: ciudad, franja, lluvia) con
**medidas** (lo que se resume: tiempo, valor). Sobre él se **filtra** (se deja una parte), se
**desglosa** (se baja a más detalle) y se **pivota** (se cambia qué va en filas y en columnas). Es
lo que hacen las tablas dinámicas de Excel, Power BI y Looker Studio.

### La tabla dinámica

`pd.pivot_table` resume una medida (`values`) por dos dimensiones (`index` y `columns`) con la
función que se le diga (`aggfunc`). `margins=True` añade los totales.

In [ ]:
# [dinamica]
tabla = pd.pivot_table(pedidos, values="tiempo_entrega_min",
                       index="ciudad", columns="lluvia",
                       aggfunc="median", observed=True,
                       margins=True, margins_name="Total")
tabla.round(1)

In [ ]:
# [efecto_lluvia]
# Cuántos minutos suma la lluvia, ciudad por ciudad (con la mediana)
(tabla["Sí"] - tabla["No"]).round(1)

En el total, Bogotá tarda 9,4 min más que Barranquilla; sin lluvia, 8,1. ¿Por qué se acorta la
diferencia? Porque la columna «Total» mezcla pedidos con y sin lluvia, y en Bogotá llueve en más
pedidos:

In [ ]:
# [composicion]
con_lluvia = (pedidos["lluvia"] == "Sí").groupby(ciudad).mean()
(con_lluvia * 100).round(1)

### Desglosar: los lentos, por distancia y lluvia

Partimos la distancia en tramos con cortes explícitos. El 4,4 es $Q_3$ de la distancia. El 7,7 es
su valla superior, y 26 pedidos la tienen exacta: con `right=True` quedan en el tramo del medio, no
entre los lejanos. Un pedido es **lento** si pasa la valla de 59,15 min; la media de una columna de
100 y 0 es el porcentaje.

In [ ]:
# [desglose]
tramo = pd.cut(distancia, bins=[0, 4.4, 7.7, 14], right=True,
               labels=["hasta 4,4 km", "4,4 a 7,7 km", "más de 7,7 km"])
lento = (tiempo > valla_sup) * 100     # 100 si es atípico lento, 0 si no
pd.pivot_table(pedidos.assign(tramo=tramo, lento=lento),
               values="lento", index="tramo", columns="lluvia",
               aggfunc="mean", observed=True).round(1)

## 4 · Para tu tablero

El proyecto final es un tablero semanal para la gerencia (el
[enunciado](https://jotamao1985.github.io/ANaLISIS-ESTADiSTICO-DE-LA-INFORMACIoN_Ucomp/proyecto-tablero.html)
tiene todos los detalles). El primer paso es agrupar por **semana ISO**, que va de lunes a domingo,
y marcar las semanas que no están completas. Esta celda no tiene verificador: los indicadores y las
gráficas son el trabajo de tu pareja.

In [ ]:
# [semanas]
fecha = pd.to_datetime(pedidos["fecha"])
# Semana ISO: de lunes a domingo
semana = fecha.dt.isocalendar().week.rename("semana")
por_semana = pd.DataFrame({
    "pedidos": pedidos.groupby(semana).size(),
    "dias": fecha.groupby(semana).nunique(),   # días con pedidos
})
por_semana["completa"] = por_semana["dias"] == 7
print("Semanas:", len(por_semana))
por_semana[~por_semana["completa"]]          # las incompletas

---
# Práctica guiada

Tres ejercicios; el tercero es **opcional**, para la casa. Cada ejercicio termina con un verificador
(la caja del ejercicio 1 no lo tiene) que dice qué está bien y, en lo demás, te da una pista sin
darte la respuesta. Reemplaza cada `...` y ejecuta; puedes repetir la celda las veces que quieras.
Las respuestas escritas van al final, en «Tus respuestas»: esas no las revisa ningún verificador.

## Ejercicio 1 · Los atípicos del valor del pedido

Calcula las vallas de 1,5 RIC de `valor_pedido` y cuenta los pedidos **estrictamente** fuera de
ellas (`n_ric`). Después cuenta los que tienen $|z| > 3$ (`n_z`). Deja como está la llamada final a
`revisar_atipicos`: entrega cada valor con su nombre. Pista: las celdas [vallas] y [puntaje_z].

Después responde abajo: ¿por qué z marca menos pedidos que las vallas?

In [ ]:
# [ej1]
valor = pedidos["valor_pedido"]
valla_inferior = ...   # Q1 - 1.5 RIC
valla_superior = ...   # Q3 + 1.5 RIC
n_ric = ...            # pedidos estrictamente fuera de las vallas
n_z = ...              # pedidos con |z| > 3
revisar_atipicos(valla_inferior=valla_inferior,
                 valla_superior=valla_superior, n_ric=n_ric, n_z=n_z)

Ahora dibuja la caja de `valor` con una línea en `valla_superior`. Este paso **no** tiene verificador.
Pista: la celda [caja_tiempo], incluidas las líneas que imprimen los bigotes. Responde abajo: ¿dónde termina el bigote superior? ¿Coincide con la
valla? ¿Por qué el pedido que vale justo la valla no es un punto suelto?

In [ ]:
# [ej1_caja]
# Dibuja aquí la caja de valor, con una línea en valla_superior

## Ejercicio 2 · Una tabla dinámica: ciudad por franja

Construye la tabla dinámica de la **mediana** del tiempo de entrega, con las ciudades en las filas y
las franjas en las columnas. Pista: la celda [dinamica]. Si pandas responde «agg function failed»,
revisa `values=`: sin él, intenta resumir todas las columnas.

Después responde abajo: ¿la franja más lenta es la misma en todas las ciudades?

In [ ]:
# [ej2]
tabla_franja = ...   # pd.pivot_table(pedidos, values=..., index=..., columns=...,
                     #                aggfunc=..., observed=True)
revisar_dinamica(tabla_franja)

## Ejercicio 3 (opcional) · ¿Y si el tope escondiera rutas de 30 km?

Los 20 pedidos con 14 km son «14 o más». Supón que en realidad eran de 30 km: haz una **copia** de
`distancia_km` con 30 en lugar de 14, sin cambiar `pedidos`, y calcula su mediana y su media. Pista:
`replace` cambia un valor por otro y devuelve una copia. Deja la última línea como está.

Después responde abajo: ¿cambia la mediana? ¿Y la media? ¿Qué te dice eso de un dato censurado?

In [ ]:
# [ej3]
distancia_30 = ...   # una copia de distancia_km con 30 en lugar de 14
mediana_30 = ...     # su mediana
media_30 = ...       # su media
revisar_tope(mediana_30=mediana_30, media_30=media_30)

### Tus respuestas

*Doble clic para escribir aquí.*

1. Ejercicio 1: z marca menos pedidos que las vallas porque …; el bigote superior termina en …, que
   (no) coincide con la valla porque …; el pedido que vale justo la valla no es un punto suelto
   porque …
2. Ejercicio 2: la franja más lenta es … en …; (no) es la misma en todas porque …
3. Ejercicio 3: la mediana …; la media …; de un dato censurado aprendo que …
4. Autoevaluación. Para cada pedido, di su causa (error, real o censurado) y si es atípico respecto
   a todos, solo respecto a su grupo o a ninguno:
   - un pedido de 62 min en Bogotá (la valla de Bogotá es de 63,40 min): …
   - una ruta registrada con 14,0 km: …
   - un pedido de farmacia de 120 000 pesos (la valla de las farmacias es de 103 000; la de todos
     los pedidos, de 155 600): …
   - un tiempo de entrega de −5 min: …
   - un pedido de una empresa con 44 productos: …